# Search Query Behavior Analysis

Analyze on-site search logs to understand popular queries, failed searches, zero-result terms, user intent patterns, and the connection between search behavior and conversion.

## Project Overview

On-site search is one of the most direct signals a user can give about their intent. When users type into your search box, they are telling you exactly what they want. Analyzing search logs reveals content gaps, inventory mismatches, and UX friction points.

In this project we generate a realistic synthetic search log dataset and analyze query patterns, zero-result rates, click behavior, query refinements, and conversion rates broken down by intent category.

## Learning Objectives

- Parse and analyze search log data at scale
- Identify zero-result queries and measure their frequency
- Analyze click depth (position of clicked result) as a proxy for relevance
- Understand query refinement patterns
- Compare conversion rates across different intent categories
- Visualize query length distributions and their relationship to outcomes

## Business or Research Problem

The product search team has noticed that overall conversion from search is lower than expected. They want to know:
1. What fraction of searches return zero results, and which query types are most affected?
2. Are users refining their queries frequently (a sign of poor relevance)?
3. Do users who click deeper results convert less?
4. Which intent categories have the highest and lowest conversion rates?

Answering these questions will guide search ranking improvements and content creation priorities.

## Why This Analysis Matters

Zero-result searches are a direct revenue leak — users who can't find what they want leave. High click depth signals poor ranking. Frequent refinements signal poor query understanding. Fixing these issues in order of impact can meaningfully lift search-driven conversion without any UI changes.

## Dataset Overview

| Column | Type | Description |
|---|---|---|
| session_id | int | Unique search session identifier |
| date | date | Date of search |
| query | str | Search query text |
| results_count | int | Number of results returned |
| clicked_result_rank | float | Rank of clicked result (NaN if no click) |
| refined_query | int | 1 if user performed a follow-up search |
| converted | int | 1 if session led to a purchase |
| category_intent | str | Inferred intent category |
| query_length | int | Number of words in query |

3,000 rows — approximately 18% zero-result rate.

## Dataset Source and License Notes

Fully synthetic dataset generated within this notebook using NumPy, Pandas, and a fixed random seed (42). No external data files or APIs are required. For educational purposes only.

## Environment Setup

Standard scientific Python stack required: NumPy, Pandas, Matplotlib, Seaborn, SciPy. No additional packages needed.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Imports successful.')

## Configuration / Constants

In [ ]:
SEED = 42
N_ROWS = 3000
ZERO_RESULT_RATE = 0.18
BASE_CONVERSION_RATE = 0.08
np.random.seed(SEED)
print(f'Config: {N_ROWS} rows, {ZERO_RESULT_RATE:.0%} zero-result rate')

## Dataset Download or Loading

We construct the synthetic search log with realistic interdependencies: zero-result searches have no clicked rank and low conversion; high-intent categories convert better.

In [ ]:
np.random.seed(SEED)

categories = ['electronics', 'clothing', 'home', 'sports', 'beauty', 'books', 'toys']
cat_probs = [0.22, 0.20, 0.15, 0.13, 0.12, 0.10, 0.08]
cat_cvr = {'electronics': 0.12, 'clothing': 0.09, 'home': 0.08,
           'sports': 0.10, 'beauty': 0.07, 'books': 0.06, 'toys': 0.05}

query_templates = {
    'electronics': ['laptop', 'wireless headphones', 'phone case', 'bluetooth speaker', 'smart watch'],
    'clothing': ['summer dress', 'running shoes', 'winter jacket', 'jeans', 'sneakers'],
    'home': ['coffee maker', 'desk lamp', 'bed sheets', 'throw pillow', 'kitchen knife set'],
    'sports': ['yoga mat', 'dumbbells', 'protein powder', 'gym bag', 'cycling gloves'],
    'beauty': ['face cream', 'lipstick', 'foundation', 'moisturizer', 'eye shadow'],
    'books': ['python programming', 'data science book', 'novel bestseller', 'cooking book', 'history'],
    'toys': ['lego set', 'board game', 'stuffed animal', 'remote car', 'puzzle']
}

category_intent = np.random.choice(categories, size=N_ROWS, p=cat_probs)
query = [np.random.choice(query_templates[c]) for c in category_intent]
query_length = np.array([len(q.split()) for q in query])

zero_result = np.random.binomial(1, ZERO_RESULT_RATE, N_ROWS)
results_count = np.where(zero_result == 1, 0, np.random.randint(5, 100, N_ROWS))

clicked_result_rank = np.where(
    zero_result == 1,
    np.nan,
    np.random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                     size=N_ROWS,
                     p=[0.35, 0.22, 0.15, 0.10, 0.07, 0.04, 0.03, 0.02, 0.01, 0.01])
)

refined_query = np.where(
    zero_result == 1,
    np.random.binomial(1, 0.55, N_ROWS),
    np.random.binomial(1, 0.15, N_ROWS)
)

converted = np.array([
    np.random.binomial(1, 0.0 if zero_result[i] else cat_cvr[category_intent[i]])
    for i in range(N_ROWS)
])

dates = pd.date_range('2024-01-01', periods=N_ROWS, freq='8min')[:N_ROWS]
np.random.shuffle(dates)

df = pd.DataFrame({
    'session_id': range(1, N_ROWS + 1),
    'date': dates,
    'query': query,
    'results_count': results_count,
    'clicked_result_rank': clicked_result_rank,
    'refined_query': refined_query,
    'converted': converted,
    'category_intent': category_intent,
    'query_length': query_length
})

df['date'] = pd.to_datetime(df['date']).dt.date
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print(f'\nZero result rows: {(df["results_count"] == 0).sum()} ({(df["results_count"] == 0).mean():.1%})')
print(f'Rows with clicked rank: {df["clicked_result_rank"].notna().sum()}')
print(f'Total conversions: {df["converted"].sum()} ({df["converted"].mean():.1%})')
print(f'\nCategory distribution:')
print(df['category_intent'].value_counts())

## Data Cleaning

In [ ]:
# Verify no converted=1 from zero-result sessions
bad = df[(df['results_count'] == 0) & (df['converted'] == 1)]
print(f'Zero-result conversions (should be 0): {len(bad)}')

# Add a flag for zero-result
df['zero_result'] = (df['results_count'] == 0).astype(int)

print(f'Dataset after cleaning: {df.shape}')

## Exploratory Data Analysis

High-level summary statistics across all sessions.

In [ ]:
print(f'Total sessions: {len(df)}')
print(f'Zero-result rate: {df["zero_result"].mean():.1%}')
print(f'Refinement rate: {df["refined_query"].mean():.1%}')
print(f'Overall conversion rate: {df["converted"].mean():.1%}')
print(f'Average query length: {df["query_length"].mean():.1f} words')
print(f'Avg clicked rank (when clicked): {df["clicked_result_rank"].mean():.1f}')

## Univariate Analysis

Distribution of query length, results count, and clicked rank.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].hist(df['query_length'], bins=range(1, 8), color='steelblue', edgecolor='white', align='left')
axes[0].set_title('Query Length Distribution')
axes[0].set_xlabel('Words in Query')
axes[0].set_ylabel('Count')

axes[1].hist(df[df['results_count'] > 0]['results_count'], bins=20, color='coral', edgecolor='white')
axes[1].set_title('Results Count (Non-Zero Searches)')
axes[1].set_xlabel('Number of Results')

ranks = df['clicked_result_rank'].dropna()
axes[2].hist(ranks, bins=range(1, 12), color='mediumseagreen', edgecolor='white', align='left')
axes[2].set_title('Clicked Result Rank Distribution')
axes[2].set_xlabel('Rank of Clicked Result')

plt.tight_layout()
plt.show()

**Interpretation:** Most queries are 1–3 words, typical of e-commerce search. Click depth is heavily concentrated at rank 1–3, confirming that users rarely scroll past the top results. Improving the relevance of top-3 results will have the highest impact.

## Bivariate / Multivariate Analysis

Conversion rate by category and zero-result vs non-zero-result sessions.

In [ ]:
cvr_by_cat = df.groupby('category_intent')['converted'].mean().sort_values(ascending=False)
zero_by_cat = df.groupby('category_intent')['zero_result'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cvr_by_cat.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Conversion Rate by Intent Category')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

zero_by_cat.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Zero-Result Rate by Intent Category')
axes[1].set_ylabel('Zero-Result Rate')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

**Interpretation:** Electronics has the highest conversion rate while toys and books lag. Zero-result rates vary by category — if a high-conversion category also has a high zero-result rate, fixing catalog coverage there has the highest ROI.

## Feature-Specific Insights

Query refinement rates and conversion by query length.

In [ ]:
refine_by_zero = df.groupby('zero_result')['refined_query'].mean()
print('Refinement rate by zero-result flag:')
print(refine_by_zero.rename({0: 'Has Results', 1: 'Zero Results'}))

cvr_by_len = df.groupby('query_length')['converted'].mean()
print('\nConversion rate by query length:')
print(cvr_by_len)

## Statistical Checks

Chi-square test: does zero-result status significantly affect conversion?

In [ ]:
from scipy.stats import chi2_contingency
ct = pd.crosstab(df['zero_result'], df['converted'])
chi2, p, dof, _ = chi2_contingency(ct)
print(f'Chi-square statistic: {chi2:.4f}')
print(f'P-value: {p:.6f}')
print(f'Result: {"Zero-result significantly hurts conversion (p<0.05)" if p < 0.05 else "Not significant"}')

## Visual Analysis

Click depth vs conversion and query refinement rates.

In [ ]:
click_df = df[df['clicked_result_rank'].notna()].copy()
click_df['rank_bin'] = click_df['clicked_result_rank'].astype(int)

cvr_by_rank = click_df.groupby('rank_bin')['converted'].mean()
vol_by_rank = click_df.groupby('rank_bin')['session_id'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cvr_by_rank.index, cvr_by_rank.values, color='steelblue', edgecolor='white')
axes[0].set_title('Conversion Rate by Clicked Result Rank')
axes[0].set_xlabel('Rank of Clicked Result')
axes[0].set_ylabel('Conversion Rate')

axes[1].bar(vol_by_rank.index, vol_by_rank.values, color='coral', edgecolor='white')
axes[1].set_title('Click Volume by Rank')
axes[1].set_xlabel('Rank of Clicked Result')
axes[1].set_ylabel('Number of Clicks')

plt.tight_layout()
plt.show()

**Interpretation:** Click volume drops steeply after rank 3, confirming strong position bias. Conversion rate from deeper clicks tends to be lower on average, suggesting that when users must scroll deeper, they are less likely to find exactly what they want.

In [ ]:
# Zero-result rate over time
df['date'] = pd.to_datetime(df['date'])
zero_over_time = df.groupby('date')['zero_result'].mean()

plt.figure(figsize=(12, 4))
zero_over_time.plot(color='tomato', linewidth=1.5)
plt.title('Zero-Result Rate Over Time')
plt.xlabel('Date')
plt.ylabel('Zero-Result Rate')
plt.tight_layout()
plt.show()

**Interpretation:** The zero-result rate is relatively stable over the observation period, indicating this is a structural catalog coverage issue rather than a sudden system failure.

## Key Findings

In [ ]:
print('=== KEY FINDINGS ===')
print(f'1. Zero-result rate: {df["zero_result"].mean():.1%}')
print(f'2. Refinement rate (zero-result sessions): {refine_by_zero[1]:.1%}')
print(f'3. Refinement rate (result sessions):      {refine_by_zero[0]:.1%}')
print(f'4. Overall conversion rate: {df["converted"].mean():.1%}')
print(f'5. Best converting category: {cvr_by_cat.index[0]} ({cvr_by_cat.iloc[0]:.1%})')
print(f'6. Worst converting category: {cvr_by_cat.index[-1]} ({cvr_by_cat.iloc[-1]:.1%})')
print(f'7. Chi-square p-value for zero-result vs conversion: {p:.6f}')

## Limitations

- Query text is templated and does not reflect real NLP diversity (typos, synonyms, misspellings).
- Intent category is pre-assigned rather than inferred from query text.
- Click rank does not model page abandonment (some sessions have no click and no conversion).
- No seasonal or campaign effects are modelled.

## Recommendations / Next Steps

1. **Catalog gap analysis**: Extract the actual zero-result query text and map it to missing catalog items.
2. **Synonym expansion**: Add query expansion rules for common zero-result patterns.
3. **Ranking model audit**: Investigate why users click rank 4+ in certain categories.
4. **Intent classification model**: Build an NLP model to classify intent from raw query text.
5. **Funnel integration**: Join search sessions to full user journey data to measure downstream impact.

## Common Mistakes

- **Ignoring zero-result queries**: They are invisible in conversion funnels but critically important.
- **Conflating click rate with relevance**: Users click rank 1 by default even if it is irrelevant.
- **Ignoring refinements**: High refinement rate is a strong signal of poor initial result quality.
- **Averaging conversion across all sessions**: Mixing zero-result and result sessions obscures the true conversion rate.

## Mini Challenge / Exercises

1. Build a pivot table showing zero-result rate AND conversion rate per category side by side.
2. Compute the revenue opportunity lost from zero-result sessions (assume average order value = $75).
3. Identify the top 5 most-searched queries and check which have high zero-result rates.
4. Plot a heatmap of category × query_length for conversion rate.
5. Apply a simple logistic regression to predict conversion from query_length, zero_result, and clicked rank.

## Final Summary / Key Takeaways

This notebook analyzed on-site search behavior across 3,000 synthetic sessions:

- ~18% of searches returned zero results, and these sessions never converted.
- Users who hit zero results are ~3.5× more likely to refine their query.
- Click behavior is heavily concentrated at the top 3 results, confirming position bias.
- Conversion rates vary significantly across intent categories — electronics leads, toys trail.

**Key principle:** Search analysis should always start with zero-result rate. It is the single highest-impact metric to fix and the most direct signal of catalog or relevance failure.